# Fake News Detection with LSTM & Bidirectional LSTM

**Final-year Deep Learning project — exploratory notebook**

This notebook documents the complete modelling workflow used in this project:

1. Loading the Kaggle *Fake and Real News* dataset (`Fake.csv`, `True.csv`)
2. Exploratory data analysis and duplicate inspection
3. Text cleaning + tokenisation (NLTK stopwords, Keras `Tokenizer`)
4. Leakage-free 70/15/15 train–validation–test split
5. LSTM and BiLSTM classifiers in TensorFlow / Keras
6. Test-set evaluation: accuracy, precision, recall, F1, ROC-AUC, confusion matrices
7. Comparison and interpretation

> The production training entry point is `train_model.py`; this notebook reuses the
> same modules (`config.py`, `preprocess.py`, `evaluation.py`) so results are consistent.
> Set `EPOCHS = 2` for a quick notebook demo; the shipped models were trained with
> early stopping and best-checkpoint restore.


## 1. Environment setup

Run from the `notebooks/` directory (or the project root — the cell handles both).

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import config
from preprocess import load_raw_dataset, load_clean_dataset, split_dataset, TextPreprocessor, clean_text
from evaluation import compute_metrics, plot_training_curves, plot_confusion_matrix
from train_model import build_lstm_model, build_bilstm_model

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## 2. Dataset

| File | Meaning | Rows (raw) |
|---|---|---|
| `dataset/Fake.csv` | Fabricated / unreliable articles | 23,481 |
| `dataset/True.csv` | Reuters-style reporting | 21,417 |

Columns: `title`, `text`, `subject`, `date`. Labels are created by the loader
(`0 = FAKE`, `1 = REAL`).

In [ ]:
raw = load_raw_dataset()

print(f"Combined rows      : {len(raw):,}")
print(f"Fake (label=0)     : {(raw.label == 0).sum():,}")
print(f"Real (label=1)     : {(raw.label == 1).sum():,}")
print(f"Duplicate rows dropped: {raw.attrs.get('duplicates_removed', 0):,}")
raw[["title", "text", "subject", "label"]].head(3)

### 2.1 Why deduplication matters

The Kaggle `Fake.csv` file contains **5,573 duplicated title+text pairs**. Leaving them in
inflates the effective weight of repeated articles and can leak evaluation examples into
training. `load_raw_dataset()` removes exact duplicates before splitting.

In [ ]:
lengths = raw["content"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(lengths, bins=60, color="#2563eb", alpha=.85)
axes[0].set_title("Article length (raw words)"); axes[0].set_xlabel("words"); axes[0].set_ylabel("articles")
axes[1].bar(["Fake", "Real"], [(raw.label == 0).sum(), (raw.label == 1).sum()],
            color=["#ef4444", "#10b981"])
axes[1].set_title("Class balance"); axes[1].set_ylabel("articles")
plt.tight_layout(); plt.show()

lengths.describe().round(1)

## 3. Text preprocessing

The cleaning function is **stateless** (no corpus statistics), so it is safe to apply
before splitting:

- strip HTML tags and decode entities
- remove URLs, emails and the Reuters byline
- unicode normalisation, lowercasing
- punctuation and digit removal
- regex tokenisation, NLTK stopword removal (negations kept)

In [ ]:
example = raw.loc[0, "content"]
print("BEFORE:\n", example[:320], "\n")
print("AFTER:\n", clean_text(example)[:320])

In [ ]:
df = load_clean_dataset()
print(f"Usable articles after cleaning: {len(df):,}")
print(f"Fake: {(df.label == 0).sum():,} | Real: {(df.label == 1).sum():,}")
df[["clean", "label"]].head(2)

## 4. Leakage-free split and tokenisation

`train_test_split` is stratified. The tokenizer is fitted **only on the training
fraction**; validation and test text never influence the vocabulary.

In [ ]:
train_df, val_df, test_df = split_dataset(df)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

preprocessor = TextPreprocessor().fit(train_df["clean"].tolist())
print(f"Vocabulary size: {preprocessor.vocab_size:,} (max_words={config.MAX_WORDS})")

X_train = preprocessor.transform(train_df["clean"])
X_val = preprocessor.transform(val_df["clean"])
X_test = preprocessor.transform(test_df["clean"])
y_train = train_df["label"].to_numpy().astype("float32")
y_val = val_df["label"].to_numpy().astype("float32")
y_test = test_df["label"].to_numpy().astype("float32")

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
token_counts = train_df["clean"].str.split().str.len()
print(token_counts.describe().round(1))
print(f"Articles truncated to {config.MAX_LEN} tokens: {(token_counts > config.MAX_LEN).mean():.1%}")

## 5. Model 1 — LSTM

```
Embedding(20,000 × 128) → SpatialDropout(0.2) → LSTM(64)
→ Dropout(0.4) → Dense(32, ReLU) → Dropout(0.2) → Dense(1, sigmoid)
```


In [ ]:
lstm = build_lstm_model(preprocessor.vocab_size, preprocessor.max_len)
lstm.summary()

In [ ]:
EPOCHS = 2  # increase to 8 for the full run (train_model.py uses early stopping)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1,
                                      min_lr=1e-5),
]
lstm_history = lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
plot_training_curves(lstm_history.history, "lstm",
                     config.IMAGES_DIR / "notebook_training_lstm.png");
print("saved:", config.IMAGES_DIR / "notebook_training_lstm.png")

In [ ]:
lstm_probs = lstm.predict(X_test, batch_size=config.BATCH_SIZE, verbose=0).ravel()
lstm_metrics = compute_metrics(y_test, lstm_probs)
print({k: round(lstm_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]})
plot_confusion_matrix(lstm_metrics, "lstm", config.IMAGES_DIR / "notebook_cm_lstm.png")

## 6. Model 2 — Bidirectional LSTM

The BiLSTM runs two LSTM passes (forward + backward) over the same embedding and
concatenates the final states, giving each token context from both directions.

In [ ]:
bilstm = build_bilstm_model(preprocessor.vocab_size, preprocessor.max_len)
bilstm.summary()

In [ ]:
bilstm_history = bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)
plot_training_curves(bilstm_history.history, "bilstm",
                     config.IMAGES_DIR / "notebook_training_bilstm.png")

In [ ]:
bilstm_probs = bilstm.predict(X_test, batch_size=config.BATCH_SIZE, verbose=0).ravel()
bilstm_metrics = compute_metrics(y_test, bilstm_probs)

comparison = pd.DataFrame([
    {"Model": "LSTM", **{k: round(lstm_metrics[k], 4) for k in
        ["accuracy", "precision", "recall", "f1_score", "roc_auc"]}},
    {"Model": "BiLSTM", **{k: round(bilstm_metrics[k], 4) for k in
        ["accuracy", "precision", "recall", "f1_score", "roc_auc"]}},
])
comparison

## 7. Interpretation & limitations

- Both architectures reach very high test accuracy on this corpus, but the task is
  **easier than real-world fact-checking**: fake articles come largely from a set of
  alternative outlets and real articles largely from Reuters, so publisher-specific
  style cues are learnable.
- Deduplication prevents inflated scores but cannot remove near-duplicate phrasing.
- The model is English-only, truncates at 256 tokens and covers 2015–2018 vocabulary.
- A confidence score is model certainty, **not** the probability that an article is true.

The trained artifacts (`models/*.keras`, `models/tokenizer.pkl`, `models/metrics.json`)
are produced by the reproducible script:

```bash
python train_model.py --epochs 8
python evaluate_model.py
```
